# RAGTIME Single-Query Retrieval

This notebook uses `RagtimeRetriever` to send one query to the private RAGTIME `POST /pipeline` endpoint, whose successful responses are cached without expiry, then enrich every ranked ID from the revision-pinned local Hugging Face corpus index. Set `RESULT_COUNT` to control the RoutIR output count; `TOP_K = None` keeps every result returned by the endpoint. It does not call the private `/content` endpoint by default.

Decrypt the repository credentials before starting Jupyter:

```bash
sops decrypt --output .env config/secrets.sops.env
uv run ragtime-corpus build
uv run jupyter lab notebooks/ragtime_single_query.ipynb
```

Change `QUERY` below, then run the cells in order.

In [ ]:
from textwrap import shorten

import pandas as pd
from IPython.display import display
from trec_ragtime import RagtimeRetriever

RESULT_COUNT = 100
PIPELINE = f"ragtime2%{RESULT_COUNT}"
QUERY = "health benefits and risks of Nordic walking for older adults"
TOP_K = None  # Enrich every result returned by the endpoint.

In [ ]:
with RagtimeRetriever.from_env(document_backend="local") as retriever:
    result = retriever.retrieve(QUERY, pipeline=PIPELINE, top_k=TOP_K)
    shared_cache_path = retriever.cache_path
    document_backend = retriever.document_backend
    corpus_path = retriever.corpus_path
    corpus_revision = getattr(getattr(retriever.corpus, "metadata", None), "revision", None)

records = result.as_records(include_text=False)
for record, document in zip(records, result.documents, strict=True):
    record["text_preview"] = (
        shorten(" ".join(document.text.split()), width=280, placeholder="...")
        if document.text is not None
        else None
    )
results = pd.DataFrame(records)

In [ ]:
print(f"Query: {QUERY}")
print(f"Pipeline: {result.pipeline}")
print(f"Results returned: {len(results)}")
print(f"Fully hydrated: {result.fully_hydrated}")
print(f"Search cache: {'hit' if result.search_local_cache_hit else 'miss'}")
print(f"Document backend: {document_backend}")
print(f"Corpus path: {corpus_path}")
print(f"Corpus revision: {corpus_revision}")
cache_markers = [document.content_local_cache_hit for document in result.documents if document.content_local_cache_hit is not None]
if cache_markers:
    print(f"Document cache hits: {sum(cache_markers)}/{len(cache_markers)}")
else:
    print("Document cache hits: not applicable (local index)")
print(f"Shared cache path: {shared_cache_path}")
display(results)